# 01 — Revisión de datos

Objetivo: determinar qué información de Volve está disponible y qué condiciones requieren revisión antes de explorar o modelar. No se modifican originales ni se aplica limpieza automática.

**Recorrido:** carga y contexto → copias y vistas simplificadas → validaciones por pozo y flujo → síntesis de hallazgos → próximos pasos en el cuaderno 02.

Los controles específicos permiten revisar la calidad de los datos en el contexto operacional; las decisiones de limpieza se documentan al final.


In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / "data" / "raw").is_dir():
    ROOT = ROOT.parent
assert (ROOT / "data" / "raw").is_dir(), "Abrir desde la raíz del proyecto o notebooks/."
ARCHIVO = ROOT / "data" / "raw" / "Volve production data.xlsx"

# Los identificadores se conservan como texto.
diario = pd.read_excel(ARCHIVO, sheet_name="Daily Production Data", dtype={
    "NPD_WELL_BORE_CODE": "string", "NPD_FIELD_CODE": "string", "NPD_FACILITY_CODE": "string"
})
diario["DATEPRD"] = pd.to_datetime(diario["DATEPRD"], errors="raise")
mensual = pd.read_excel(ARCHIVO, sheet_name="Monthly Production Data", skiprows=[1],
                        na_values=["NULL"], dtype={"NPDCode": "string"})


## Cómo leer los nombres de las variables

Las tablas y los gráficos muestran nombres en español. En el código se conservan los nombres originales para poder rastrear cada variable hasta el Excel. Los códigos de pozos y los códigos `OP` y `WI` se conservan como identificadores; el significado de estos últimos sigue pendiente de confirmación.

La tabla de equivalencias permite relacionar ambas versiones. Las unidades y las interpretaciones pendientes no se dan por confirmadas al traducir.


In [ ]:
# Traducciones de presentación: los datos conservan sus nombres y valores originales.
etiquetas_es = {'DATEPRD': 'Fecha del registro',
 'WELL_BORE_CODE': 'Identificador textual del pozo',
 'NPD_WELL_BORE_CODE': 'Código NPD del pozo',
 'NPD_WELL_BORE_NAME': 'Nombre del pozo',
 'NPD_FIELD_CODE': 'Código del campo',
 'NPD_FIELD_NAME': 'Nombre del campo',
 'NPD_FACILITY_CODE': 'Código de instalación',
 'NPD_FACILITY_NAME': 'Nombre de instalación',
 'ON_STREAM_HRS': 'Horas operativas (h)',
 'AVG_DOWNHOLE_PRESSURE': 'Presión promedio de fondo',
 'AVG_DOWNHOLE_TEMPERATURE': 'Temperatura promedio de fondo',
 'AVG_DP_TUBING': 'Diferencia de presión promedio en la tubería',
 'AVG_ANNULUS_PRESS': 'Presión promedio del espacio anular',
 'AVG_CHOKE_SIZE_P': 'Apertura promedio de la válvula de restricción',
 'AVG_CHOKE_UOM': 'Unidad de apertura de la válvula',
 'AVG_WHP_P': 'Presión promedio en cabeza de pozo',
 'AVG_WHT_P': 'Temperatura promedio en cabeza de pozo',
 'DP_CHOKE_SIZE': 'Diferencia de presión en la válvula de restricción',
 'BORE_OIL_VOL': 'Volumen de petróleo (unidad pendiente)',
 'BORE_GAS_VOL': 'Volumen de gas (unidad pendiente)',
 'BORE_WAT_VOL': 'Volumen de agua producida (unidad pendiente)',
 'BORE_WI_VOL': 'Volumen de agua inyectada (interpretación y unidad pendientes)',
 'FLOW_KIND': 'Tipo de flujo',
 'WELL_TYPE': 'Tipo de pozo (código)',
 'Wellbore name': 'Nombre del pozo',
 'NPDCode': 'Código NPD del pozo',
 'Year': 'Año',
 'Month': 'Mes',
 'On Stream': 'Horas operativas del mes (h)',
 'Oil': 'Petróleo mensual (Sm³)',
 'Gas': 'Gas mensual (Sm³)',
 'Water': 'Agua producida mensual (Sm³)',
 'GI': 'Gas inyectado mensual (interpretación pendiente; Sm³)',
 'WI': 'Agua inyectada mensual (interpretación pendiente; Sm³)',
 'tipo': 'Tipo de dato',
 'valores_distintos': 'Valores distintos',
 'faltantes_total_pct': 'Faltantes en el total (%)',
 'faltantes_produccion_pct': 'Faltantes en producción (%)',
 'faltantes_inyeccion_pct': 'Faltantes en inyección (%)',
 'decision_vista': 'Decisión para la vista',
 'motivo': 'Motivo',
 'columna': 'Variable',
 'horas_cero': 'Registros con 0 horas operativas',
 'horas_fuera_0_24': 'Horas operativas fuera de 0 a 24',
 'petroleo_con_horas_cero': 'Petróleo positivo con 0 horas operativas',
 'production_con_tipo_WI': 'Flujo de producción con tipo de pozo WI',
 'registros': 'Registros',
 'count': 'Cantidad de datos disponibles',
 'mean': 'Promedio',
 'std': 'Desviación estándar',
 'min': 'Mínimo',
 '25%': 'Percentil 25',
 '50%': 'Mediana (percentil 50)',
 '75%': 'Percentil 75',
 'max': 'Máximo'}
tipos_es = {
    "object": "Texto o datos mixtos", "string": "Texto", "str": "Texto",
    "int64": "Número entero", "float64": "Número decimal",
    "datetime64[ns]": "Fecha", "datetime64[us]": "Fecha", "bool": "Verdadero o falso",
}

def tabla_en_espanol(tabla):
    """Crear una copia para mostrar, sin cambiar el dataframe de análisis."""
    vista = tabla.copy()
    vista = vista.rename(columns=etiquetas_es, index=etiquetas_es)
    if vista.index.name is not None:
        vista.index.name = etiquetas_es.get(vista.index.name, vista.index.name)
    if vista.columns.name is not None:
        vista.columns.name = etiquetas_es.get(vista.columns.name, vista.columns.name)
    if "Tipo de flujo" in vista.columns:
        vista["Tipo de flujo"] = vista["Tipo de flujo"].replace({
            "production": "Producción", "injection": "Inyección"
        })
    if "Tipo de dato" in vista.columns:
        vista["Tipo de dato"] = vista["Tipo de dato"].replace(tipos_es)
    return vista


In [ ]:
equivalencias = pd.DataFrame({
    "Variable en español": [etiquetas_es[c] for c in diario.columns],
    "Nombre original en Excel": diario.columns,
})
display(equivalencias)
print("Hoja mensual — equivalencias:")
display(pd.DataFrame({
    "Variable en español": [etiquetas_es[c] for c in mensual.columns],
    "Nombre original en Excel": mensual.columns,
}))


## Contenido y estructura

In [ ]:
print(f"Diario: {diario.shape[0]:,} registros, {diario.shape[1]} columnas")
print(f"Mensual: {mensual.shape[0]:,} registros, {mensual.shape[1]} columnas")
print("Período diario:", diario["DATEPRD"].min(), "a", diario["DATEPRD"].max())
display(tabla_en_espanol(diario.head()))
display(tabla_en_espanol(mensual.head()))
display(tabla_en_espanol(diario.dtypes.astype(str).rename("tipo").to_frame()))

## Diccionario original

Consultar el estado de verificación y las observaciones antes de asumir unidades.

In [ ]:
diccionario = pd.read_excel(ROOT / "data" / "raw" / "Diccionario_Volve.xlsx", sheet_name=None, header=3)
for hoja, tabla in diccionario.items():
    print(hoja)
    display(tabla.dropna(how="all"))

## Copias y vistas simplificadas para explorar

`diario` conserva las 24 columnas cargadas. Creamos `produccion` e `inyeccion` con `.copy()`, separando únicamente por `FLOW_KIND`, y luego dos vistas con menos columnas. Conservamos los índices originales para rastrear cada fila hasta `diario`; no son una clave del negocio.

La tabla siguiente muestra valores distintos (sin contar faltantes), disponibilidad global y por flujo, y la decisión para cada columna. Apartamos de las vistas sólo los metadatos de campo e instalación que efectivamente sean constantes y el identificador textual alternativo `WELL_BORE_CODE`. Conservamos el código y nombre NPD del pozo, `FLOW_KIND`, `WELL_TYPE`, las unidades y todas las mediciones, incluso si son constantes o tienen muchos faltantes. El identificador alternativo sigue disponible en las copias completas; no se usa aquí para unir tablas.

Estas vistas simplifican la lectura: no eliminan paradas ni atípicos, no imputan, no suavizan series y no son una selección de variables para modelos. Si aparece un flujo distinto o vacío, queda en `diario` y se informa para revisión. Las unidades pendientes siguen pendientes. No se exportan archivos nuevos.


In [ ]:
# Copias completas por flujo: conservan todas las columnas y los índices originales.
produccion = diario.loc[diario["FLOW_KIND"].eq("production")].copy()
inyeccion = diario.loc[diario["FLOW_KIND"].eq("injection")].copy()

resumen_columnas = pd.DataFrame({
    "tipo": diario.dtypes.astype(str),
    "valores_distintos": diario.nunique(dropna=True),
    "faltantes_total_pct": diario.isna().mean() * 100,
    "faltantes_produccion_pct": produccion.isna().mean() * 100,
    "faltantes_inyeccion_pct": inyeccion.isna().mean() * 100,
}).round(2)
resumen_columnas.index.name = "columna"
resumen_columnas["decision_vista"] = "Conservar"
resumen_columnas["motivo"] = "Identificación, contexto o medición; revisar antes de excluir."

metadatos_contexto = [
    "NPD_FIELD_CODE", "NPD_FIELD_NAME", "NPD_FACILITY_CODE", "NPD_FACILITY_NAME"
]
# Sólo apartar estos metadatos si son constantes y no tienen faltantes.
columnas_constantes = [
    c for c in metadatos_contexto
    if resumen_columnas.loc[c, "valores_distintos"] == 1 and diario[c].notna().all()
]
columnas_apartadas = ["WELL_BORE_CODE"] + columnas_constantes
resumen_columnas.loc[columnas_apartadas, "decision_vista"] = "Apartar de la vista"
resumen_columnas.loc["WELL_BORE_CODE", "motivo"] = (
    "Identificador textual alternativo; se conservan código y nombre NPD del pozo."
)
resumen_columnas.loc[columnas_constantes, "motivo"] = (
    "Metadato constante sin faltantes; permanece en las copias completas."
)
display(tabla_en_espanol(resumen_columnas))

columnas_vista = [c for c in diario.columns if c not in columnas_apartadas]
produccion_vista = produccion.loc[:, columnas_vista].copy()
inyeccion_vista = inyeccion.loc[:, columnas_vista].copy()

print("Diario completo:", diario.shape)
print("Vista de producción:", produccion_vista.shape)
print("Vista de inyección:", inyeccion_vista.shape)
print("Columnas apartadas sólo de las vistas:", [etiquetas_es[c] for c in columnas_apartadas])
otros_flujos = ~diario["FLOW_KIND"].isin(["production", "injection"])
print("Registros con otro flujo o flujo faltante, conservados en diario:", otros_flujos.sum())
display(tabla_en_espanol(produccion_vista.head()))
display(tabla_en_espanol(inyeccion_vista.head()))


## Mapa de los pozos y cobertura del calendario

Esta tabla describe todos los pozos, incluidos los de inyección. El período observado va desde el primer hasta el último registro de cada pozo: no demuestra que haya estado operando durante todo ese tiempo.

- **Días sin registro:** fechas ausentes dentro de ese período; no equivalen a producción cero.
- **Días con cero horas:** fechas con un registro que declara cero horas operativas; no confirman por sí solas la causa de una parada.
- **Días con horas faltantes:** fechas con al menos un registro sin ese dato.
- **Cobertura del calendario:** días con registro divididos por días del período observado.

Contamos fechas distintas para no confundir registros duplicados con días adicionales. Las categorías de horas podrían superponerse si hubiera varios registros por fecha. Producción e inyección se cuentan como registros y no se suman sus volúmenes.


In [ ]:
filas_mapa = []
for codigo_pozo, datos_pozo in diario.groupby("NPD_WELL_BORE_CODE", sort=True):
    inicio = datos_pozo["DATEPRD"].min()
    fin = datos_pozo["DATEPRD"].max()
    dias_periodo = len(pd.date_range(inicio, fin, freq="D"))
    dias_registrados = datos_pozo["DATEPRD"].nunique()
    filas_mapa.append({
        "Código del pozo": codigo_pozo,
        "Pozo": " / ".join(sorted(datos_pozo["NPD_WELL_BORE_NAME"].dropna().unique())),
        "Primer registro": inicio,
        "Último registro": fin,
        "Registros totales": len(datos_pozo),
        "Registros de producción": datos_pozo["FLOW_KIND"].eq("production").sum(),
        "Registros de inyección": datos_pozo["FLOW_KIND"].eq("injection").sum(),
        "Otros flujos o sin dato": (~datos_pozo["FLOW_KIND"].isin(["production", "injection"])).sum(),
        "Días del período": dias_periodo,
        "Días con registro": dias_registrados,
        "Días sin registro": dias_periodo - dias_registrados,
        "Cobertura del calendario (%)": round(100 * dias_registrados / dias_periodo, 2),
        "Días con cero horas": datos_pozo.loc[datos_pozo["ON_STREAM_HRS"].eq(0), "DATEPRD"].nunique(),
        "Días con horas faltantes": datos_pozo.loc[datos_pozo["ON_STREAM_HRS"].isna(), "DATEPRD"].nunique(),
        "Duplicados de pozo y fecha": datos_pozo.duplicated(["DATEPRD"]).sum(),
    })
mapa_pozos = pd.DataFrame(filas_mapa).sort_values("Pozo").reset_index(drop=True)
display(mapa_pozos)

print("Relación entre flujo registrado y tipo de pozo (códigos originales):")
display(pd.crosstab(diario["FLOW_KIND"], diario["WELL_TYPE"]).rename(
    index={"production": "Producción", "injection": "Inyección"}
).rename_axis(index="Flujo registrado", columns="Tipo de pozo"))
print("Duplicados por pozo y fecha:", diario.duplicated(["NPD_WELL_BORE_CODE", "DATEPRD"]).sum())
# Mantener este resumen para la síntesis posterior del notebook.
huecos = mapa_pozos.set_index("Pozo")["Días sin registro"]


## Faltantes por flujo y pozo

Comparar la disponibilidad global y en producción, y profundizar por pozo para distinguir faltantes asociados al flujo de sensores con poca cobertura. Los porcentajes globales no bastan para decidir una imputación.


In [ ]:
produccion = diario.loc[diario["FLOW_KIND"].eq("production")].copy()
display(tabla_en_espanol(pd.DataFrame({
    "faltantes_total_pct": diario.isna().mean() * 100,
    "faltantes_produccion_pct": produccion.isna().mean() * 100
}).round(2)))
variables = ["AVG_DOWNHOLE_PRESSURE", "AVG_ANNULUS_PRESS", "AVG_CHOKE_SIZE_P", "BORE_OIL_VOL"]
display(tabla_en_espanol(produccion.groupby("NPD_WELL_BORE_NAME")[variables].agg(lambda s: s.isna().mean() * 100).round(2)))

## Registros para revisar

Estas señales no prueban errores. Las horas cero pueden corresponder a paradas; los valores mayores a 24 requieren verificar el criterio de registro.

In [ ]:
controles = pd.DataFrame({
    "horas_cero": diario["ON_STREAM_HRS"].eq(0),
    "horas_fuera_0_24": diario["ON_STREAM_HRS"].lt(0) | diario["ON_STREAM_HRS"].gt(24),
    "petroleo_con_horas_cero": diario["BORE_OIL_VOL"].gt(0) & diario["ON_STREAM_HRS"].eq(0),
    "production_con_tipo_WI": diario["FLOW_KIND"].eq("production") & diario["WELL_TYPE"].eq("WI")
})
display(tabla_en_espanol(controles.sum().rename("registros").to_frame()))
display(tabla_en_espanol(diario.loc[controles.drop(columns="horas_cero").any(axis=1)]))

## Síntesis de la revisión

Esta tabla conecta las señales revisadas con evidencia calculada y el paso siguiente. Se actualiza al ejecutar el cuaderno; no autoriza a limpiar datos automáticamente. Las unidades y correlaciones siguen pendientes de interpretación.


In [ ]:
faltantes_anular = produccion.groupby("NPD_WELL_BORE_NAME")["AVG_ANNULUS_PRESS"].apply(
    lambda s: s.isna().mean() * 100
)
pozo_mayor_faltante = faltantes_anular.idxmax()
horas_produccion = produccion["ON_STREAM_HRS"]

sintesis = pd.DataFrame([
    {
        "Señal": "Faltantes de petróleo",
        "Evidencia": (
            f"Total: {diario['BORE_OIL_VOL'].isna().mean():.1%}; "
            f"producción: {produccion['BORE_OIL_VOL'].isna().mean():.1%}."
        ),
        "Interpretación / siguiente paso": "Interpretar por flujo; no completar con cero campos que pueden no aplicar."
    },
    {
        "Señal": "Disponibilidad de presión anular",
        "Evidencia": f"Mayor faltante en producción: {pozo_mayor_faltante}, {faltantes_anular.max():.1f}%.",
        "Interpretación / siguiente paso": "Revisar cobertura del sensor por período antes de seleccionar variables."
    },
    {
        "Señal": "Horas operativas",
        "Evidencia": (
            f"En producción: {horas_produccion.eq(0).sum()} registros con cero horas; "
            f"{(horas_produccion.lt(0) | horas_produccion.gt(24)).sum()} fuera de [0, 24]."
        ),
        "Interpretación / siguiente paso": "Ceros compatibles con paradas; verificar criterio de registro para valores fuera de rango."
    },
    {
        "Señal": "Petróleo y horas",
        "Evidencia": f"{controles['petroleo_con_horas_cero'].sum()} registros del diario con petróleo positivo y horas cero.",
        "Interpretación / siguiente paso": "Revisar asignación de volumen y fecha; no corregir sin evidencia."
    },
    {
        "Señal": "Clave y continuidad temporal",
        "Evidencia": (
            f"{diario.duplicated(['NPD_WELL_BORE_CODE', 'DATEPRD']).sum()} duplicados pozo-fecha; "
            f"{huecos.gt(0).sum()} pozos con días ausentes entre su inicio y fin."
        ),
        "Interpretación / siguiente paso": "Usar calendario explícito; no confundir el siguiente registro con el día siguiente."
    },
    {
        "Señal": "Coherencia de categorías",
        "Evidencia": f"{controles['production_con_tipo_WI'].sum()} registros combinan flujo de producción con el código de tipo de pozo WI.",
        "Interpretación / siguiente paso": "Revisar tipo de pozo y contexto antes de definir filtros."
    },
])
display(sintesis)


### Conclusión y continuidad

La revisión deja señales identificadas y controles reproducibles, pero no establece todavía un conjunto de datos apto para entrenamiento. No se han aplicado exclusiones ni imputaciones.

En `02_exploracion.ipynb`, revisar primero los pozos y períodos señalados: comparar producción, horas y disponibilidad de sensores. Contrastar allí las correlaciones globales antes de seleccionar variables. La reconciliación diaria/mensual y la confirmación de unidades siguen pendientes de incorporación o documentación dentro del flujo reproducible.

Registrar abajo las decisiones de limpieza propuestas, su evidencia y los pendientes. Una señal sin explicación permanece pendiente; no se transforma automáticamente en un error confirmado.


## Notas del análisis

- Unidades y definiciones pendientes:
- Inconsistencias que necesitan revisión:
- Variables utilizables por pozo:
- Criterios de limpieza propuestos y justificación:
